CLIP Standalone Model

In [1]:
!pip install colorama

In [2]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [3]:
!kaggle datasets download -d viswachaitanyasai/deepfake-dataset
!unzip deepfake-dataset.zip -d /content/dataset

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3505.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3506.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3507.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3508.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3509.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3510.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3511.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3512.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3513.jpg  
  inflating: /content/dataset/deepfake_dataset/validation/ffpp/fake/validation_3514.jpg  
  inflating: /content/dataset/deepfake_dataset/va

In [4]:
%pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-zsvk4z65
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-zsvk4z65
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
#!/usr/bin/env python
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
import clip
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import logging

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

##############################################
#         Advanced Data Augmentation         #
##############################################
from albumentations import (
    HorizontalFlip, VerticalFlip, ShiftScaleRotate, CLAHE,
    RandomRotate90, Transpose, HueSaturationValue, GaussNoise,
    Sharpen, Emboss, RandomBrightnessContrast, OneOf, Compose, RandomCrop
)
from albumentations.pytorch import ToTensorV2
from PIL import Image

def strong_aug(p=0.5):
    return Compose([
        RandomRotate90(p=0.2),
        Transpose(p=0.2),
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        OneOf([GaussNoise()], p=0.2),
        ShiftScaleRotate(p=0.2),
        OneOf([
            CLAHE(clip_limit=2),
            Sharpen(),
            Emboss(),
            RandomBrightnessContrast(),
        ], p=0.2),
        HueSaturationValue(p=0.2),
        RandomCrop(width=200, height=200, p=0.3),
    ], p=p)

def augment(aug, image):
    return aug(image=image)["image"]

class Aug(object):
    def __call__(self, img):
        aug_inst = strong_aug(p=0.9)
        img_np = np.array(img)
        augmented = augment(aug_inst, img_np)
        return Image.fromarray(augmented)

##############################################
#            Data Transforms                 #
##############################################
def get_transforms():
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    return {
        "train": transforms.Compose([
            Aug(),                            
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(p=0.2)
        ]),
        "valid": transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ]),
        "test": transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
    }

data_transforms = get_transforms()

##############################################
#            Dataset Loading                 #
##############################################
base_dataset_dir = "/content/dataset/deepfake_dataset"  
sub_datasets = ["celebdf", "dfdc", "ffpp", "realfake"]

train_datasets = []
val_datasets = []
test_datasets = []

for sub in sub_datasets:
    train_path = os.path.join(base_dataset_dir, "train", sub)
    val_path   = os.path.join(base_dataset_dir, "validation", sub)
    test_path  = os.path.join(base_dataset_dir, "test", sub)

    train_datasets.append(datasets.ImageFolder(train_path, transform=data_transforms["train"]))
    val_datasets.append(datasets.ImageFolder(val_path, transform=data_transforms["valid"]))
    test_datasets.append(datasets.ImageFolder(test_path, transform=data_transforms["test"]))

full_train_dataset = ConcatDataset(train_datasets)
full_val_dataset   = ConcatDataset(val_datasets)
full_test_dataset  = ConcatDataset(test_datasets)

train_subset_size = 20000
val_subset_size = 5000
train_indices = random.sample(range(len(full_train_dataset)), train_subset_size)
val_indices = random.sample(range(len(full_val_dataset)), val_subset_size)

train_dataset = Subset(full_train_dataset, train_indices)
val_dataset   = Subset(full_val_dataset, val_indices)
test_dataset  = full_test_dataset

batch_size = 16
num_workers = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

##############################################
#        Simple CLIP Detector Model          #
##############################################
class SimpleCLIPDetector(nn.Module):
    def __init__(self, num_classes=1, clip_model_name="ViT-B/32", freeze_clip=True):
        super(SimpleCLIPDetector, self).__init__()
        
        self.clip_model, _ = clip.load(clip_model_name, device=device)

        if freeze_clip:
            for param in self.clip_model.parameters():
                param.requires_grad = False

        embedding_dim = self.clip_model.visual.output_dim  # typically 512 for ViT-B/32

        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.clip_model.encode_image(x)
        features = features.float()  
        out = self.classifier(features)
        return out

model = SimpleCLIPDetector(num_classes=1, clip_model_name="ViT-B/32", freeze_clip=True)
model = model.to(device)
print("Simple CLIP Detector created.")

##############################################
#        Optimizer & Loss Setup              #
##############################################
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

##############################################
#            Training & Validation           #
##############################################
num_epochs = 15

for epoch in range(num_epochs):
    # Training
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    all_train_preds = []
    all_train_labels = []

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).int()
        train_correct += (preds.cpu() == labels.cpu().int()).sum().item()
        train_total += labels.size(0)
        all_train_preds.extend(torch.sigmoid(outputs).cpu().detach().numpy().flatten())
        all_train_labels.extend(labels.cpu().numpy().flatten())

    train_loss = running_loss / train_total
    train_acc = train_correct / train_total
    train_auc = roc_auc_score(all_train_labels, all_train_preds)

    # Validation
    model.eval()
    running_loss = 0.0
    val_correct = 0
    val_total = 0
    all_val_preds = []
    all_val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).int()
            val_correct += (preds.cpu() == labels.cpu().int()).sum().item()
            val_total += labels.size(0)
            all_val_preds.extend(torch.sigmoid(outputs).cpu().numpy().flatten())
            all_val_labels.extend(labels.cpu().numpy().flatten())

    val_loss = running_loss / val_total
    val_acc = val_correct / val_total
    val_auc = roc_auc_score(all_val_labels, all_val_preds)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Train AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}")

##############################################
#                Testing                     #
##############################################
model.eval()
running_loss = 0.0
test_correct = 0
test_total = 0
all_test_preds = []
all_test_labels = []

for images, labels in tqdm(test_loader, desc="Testing"):
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    running_loss += loss.item() * images.size(0)
    preds = (torch.sigmoid(outputs) > 0.5).int()
    all_test_preds.extend(torch.sigmoid(outputs).cpu().detach().numpy().flatten())
    all_test_labels.extend(labels.cpu().numpy().flatten())
    test_correct += (preds.cpu() == labels.cpu().int()).sum().item()
    test_total += labels.size(0)

test_loss = running_loss / test_total
test_acc = test_correct / test_total
test_auc = roc_auc_score(all_test_labels, all_test_preds)
conf_mat = confusion_matrix(all_test_labels, [1 if p > 0.5 else 0 for p in all_test_preds])
prec = precision_score(all_test_labels, [1 if p > 0.5 else 0 for p in all_test_preds])
rec = recall_score(all_test_labels, [1 if p > 0.5 else 0 for p in all_test_preds])
f1 = f1_score(all_test_labels, [1 if p > 0.5 else 0 for p in all_test_preds])

print("\n=== Test Evaluation Metrics ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC-ROC: {test_auc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(conf_mat)


Using device: cuda
Simple CLIP Detector created.


Epoch 1/15 - Training:   0%|          | 0/1250 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Epoch 1/15 - Validation: 100%|██████████| 313/313 [00:13<00:00, 23.75it/s]


Epoch [1/15] - Train Loss: 0.6259, Train Acc: 0.6379, Train AUC: 0.7052 | Val Loss: 0.5314, Val Acc: 0.7352, Val AUC: 0.8098


Epoch 2/15 - Training:   0%|          | 0/1250 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Epoch 2/15 - Validation: 100%|██████████| 313/313 [00:12<00:00, 24.20it/s]


Epoch [2/15] - Train Loss: 0.5730, Train Acc: 0.6956, Train AUC: 0.7688 | Val Loss: 0.4964, Val Acc: 0.7550, Val AUC: 0.8360


...

Epoch [14/15] - Train Loss: 0.4959, Train Acc: 0.7482, Train AUC: 0.8345 | Val Loss: 0.3966, Val Acc: 0.8120, Val AUC: 0.8999


Epoch 15/15 - Training:   0%|          | 0/1250 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Epoch 15/15 - Validation: 100%|██████████| 313/313 [00:12<00:00, 24.12it/s]


Epoch [15/15] - Train Loss: 0.4857, Train Acc: 0.7559, Train AUC: 0.8426 | Val Loss: 0.3981, Val Acc: 0.8114, Val AUC: 0.9006


Testing: 100%|██████████| 522/522 [00:20<00:00, 25.66it/s]


=== Test Evaluation Metrics ===
Test Loss: 0.3988
Test Accuracy: 0.8164
Test AUC-ROC: 0.9015
Precision: 0.7762
Recall: 0.8495
F1 Score: 0.8112
Confusion Matrix:
[[3521  949]
 [ 583 3291]]
